# W86 / #31 — MoE Substrate Closure on Colab Pro

Closes the last open P1 (`#31 MoE / Mixture-of-Experts Substrate`) on a real open-weight MoE model. Default: **`allenai/OLMoE-1B-7B-0924-Instruct`** (7 B total / 1.3 B active / 8 experts top-2) — fits easily on A100-40GB at bf16.

## One-time setup (do this before *Run all*)

1. **Runtime → Change runtime type → A100 GPU** (V100 / L4 also work; T4 is borderline).
2. **Set the HF token Colab Secret:** click the 🔑 key icon in the left sidebar → *+ Add new secret* → name = `hf_token`, value = `hf_xxxxxxxx`, toggle *Notebook access* **on**.  
   - OLMoE is NOT a gated repo, so the token is only used to avoid HF rate limits; you can also use a token created for the W86 frontier-closure work.
3. *Runtime → Run all*. Total wall-clock: ~5 min on A100 (~3 min model load + ~30 s bench).

## Output

* `moe_substrate_closure_report.json` is written to `/content/w86_moe_<TIMESTAMP>/` and copied to `/content/drive/MyDrive/coordpy_moe_closure/w86_moe_<TS>/`.
* The notebook prints a per-DoD-bullet PASS/FAIL summary from `verify_w86_moe_audit_chain.py`.

## Anti-cheat

* The closure code lives in `coordpy/moe_runtime_substrate_v1.py` on `origin/main`; this notebook only invokes it.
* HF token never leaves Colab Secrets.
* Every CID in the report re-derives offline.

In [ ]:
# --- 1. Environment probe + workdir ---
import os, sys, subprocess, json, datetime
RUN_TS = datetime.datetime.now(
    tz=datetime.timezone.utc).strftime('%Y%m%dT%H%M%SZ')
OUT_DIR = f'/content/w86_moe_{RUN_TS}'
os.makedirs(OUT_DIR, exist_ok=True)
print('RUN_TS:', RUN_TS)
print('OUT_DIR:', OUT_DIR)
subprocess.run(['nvidia-smi'], check=False)
import torch
print('torch:', torch.__version__,
      'cuda:', torch.cuda.is_available())
if torch.cuda.is_available():
    p = torch.cuda.get_device_properties(0)
    print(f'GPU: {p.name}  '
          f'mem={p.total_memory / (1024**3):.2f} GiB  '
          f'sm={p.major}.{p.minor}')

In [ ]:
# --- 2. Install transformers + accelerate + hf_hub ---
subprocess.run([
    sys.executable, '-m', 'pip', 'install', '-q',
    'transformers>=4.45.0',
    'accelerate>=0.34.0',
    'huggingface_hub>=0.24.0',
    'numpy>=1.24',
], check=True)
import transformers, accelerate, huggingface_hub
print('transformers:', transformers.__version__)
print('accelerate:', accelerate.__version__)
print('huggingface_hub:', huggingface_hub.__version__)

In [ ]:
# --- 3. HF login via Colab Secrets ---
from google.colab import userdata  # type: ignore
hf_token = userdata.get('hf_token').strip()
assert hf_token.startswith('hf_'), (
    'Colab Secret `hf_token` must start with hf_. '
    'Add it via the key icon in the left sidebar.')
os.environ['HF_TOKEN'] = hf_token
os.environ['HUGGING_FACE_HUB_TOKEN'] = hf_token
from huggingface_hub import login
login(token=hf_token, add_to_git_credential=False)
print('HF login OK; token len =', len(hf_token))

In [ ]:
# --- 4. Clone CoordPy + install in editable mode ---
import shutil
shutil.rmtree('/content/CoordPy', ignore_errors=True)
subprocess.run([
    'git', 'clone', '--depth=1',
    'https://github.com/adotdong29/CoordPy.git',
    '/content/CoordPy',
], check=True)
os.chdir('/content/CoordPy')
print('HEAD =', subprocess.check_output(
    ['git', 'log', '-1', '--oneline'], text=True).strip())
subprocess.run([
    sys.executable, '-m', 'pip', 'install', '-q', '-e', '.',
], check=True)
import coordpy
print('coordpy:', coordpy.__version__,
      'SDK:', getattr(coordpy, 'SDK_VERSION', '?'))

In [ ]:
# --- 5. Run the MoE closure driver ---
MOE_MODEL = 'allenai/OLMoE-1B-7B-0924-Instruct'
MOE_LOG = os.path.join(OUT_DIR, 'moe_run.log')
print('MOE_MODEL:', MOE_MODEL)
print('MOE_LOG:  ', MOE_LOG)
!cd /content/CoordPy && python scripts/run_w86_moe_substrate_closure.py \
    --out-dir {OUT_DIR} \
    --model-name {MOE_MODEL} \
    --device cuda:0 \
    --precision-tier tier_bf16 \
    --prompt-max-len 24 \
    --n-continuation-tokens 4 \
    --inject-layer 4 \
    --inject-magnitude 1.0 2>&1 | tee {MOE_LOG}
print()
print('=== OUT_DIR after driver ===')
!ls -la {OUT_DIR}

In [ ]:
# --- 6. Offline re-verifier (print PASS/FAIL per DoD bullet) ---
RPT = os.path.join(
    OUT_DIR, 'moe_substrate_closure_report.json')
!cd /content/CoordPy && python scripts/verify_w86_moe_audit_chain.py \
    --report {RPT}

In [ ]:
# --- 7. Print headline numbers ---
with open(RPT) as fh:
    overall = json.load(fh)
rd = overall.get('closure_31', {})
if not rd:
    print('closure_31 missing; check the log above:')
    print(overall.get('closure_31_error', '(no error field)'))
else:
    print('=== W86 #31 MoE substrate closure headline ===')
    for k in ('model_name', 'precision_tier',
              'moe_block_class_name', 'gate_class_name',
              'gate_returns_tuple', 'hook_fires_per_forward',
              'n_moe_layers', 'n_experts', 'top_k',
              'n_layers_routing_captured',
              'forward_routing_captured',
              'replay_with_routing_matches_forward_floor',
              'moe_routing_is_load_bearing',
              'routing_deterministic_across_two_forwards',
              'hidden_state_intercept_on_moe_block_moves_cid',
              'tier_tolerance',
              'max_abs_diff_with_routing_vs_forward_last_logits',
              'max_abs_diff_without_routing_vs_forward_last_logits',
              'max_abs_diff_force_random_vs_forward_last_logits',
              'wall_clock_seconds', 'bench_cid'):
        if k in rd:
            print(f'  {k}: {rd[k]}')

In [ ]:
# --- 8. Save results to Drive + download zip backup ---
from google.colab import drive  # type: ignore
drive.mount('/content/drive', force_remount=False)
DEST = (
    f'/content/drive/MyDrive/coordpy_moe_closure/'
    f'w86_moe_{RUN_TS}')
os.makedirs(DEST, exist_ok=True)
subprocess.run(
    f'cp -r {OUT_DIR}/. {DEST}/', shell=True, check=False)
print('saved to Drive:', DEST)
subprocess.run(f'ls -la {DEST}', shell=True, check=False)
zip_path = f'/content/w86_moe_{RUN_TS}.zip'
subprocess.run(
    ['zip', '-rq', zip_path, f'w86_moe_{RUN_TS}'],
    cwd='/content', check=False)
print('zip ready at', zip_path,
      f'({os.path.getsize(zip_path)/1024:.1f} KiB)')
from google.colab import files  # type: ignore
files.download(zip_path)